In [14]:
import sys
import os

try:
    current_dir = os.path.dirname(os.path.abspath(__file__))
    clients_path = os.path.join(current_dir, "../..", "src", "clients")
    sys.path.insert(0, os.path.abspath(clients_path))
except NameError:
    current_dir = os.getcwd()
    base_path = os.path.abspath(os.path.join(current_dir, "..", "..", "src", "clients"))
    sys.path.insert(0, base_path)
except Exception as e:
    print(e)

from kraken_python_client import KrakenPythonClient

client = KrakenPythonClient()

Use this notebook as a way to test and interact with the api

In [51]:
class settings:
    ##
    ##  Pair & Reference Pair
    ##
    pair = 'EURQ/USD'
    ref_pair = 'EUR/USD'

    ##
    ##  Portfolio 
    ## 
    max_capital = 200 # In USD
    
    
    ##
    ## Order Sizing
    ##
    min_order = 0.05 # 5% of max capital
    max_order = 0.10 # 10% of max capital
    

    ##
    ## Ladder Options
    ##
    ladder_increment = 0.0001
    buffer = 0.000
    ladder_size = 7

    ##
    ## Order Pricing
    ##
    ask_spread = 10     # In bips
    bid_spread = 10     # In bips
    trade_increment = 0.0001

    ##
    ##  Program Settings
    ##
    version = "v0.1.0"


In [52]:
## Strategy Logic Handler (How to price / position)
## Settings (Settings for the strategy)
## Trader (Actually trades this shit)
## Optimal Pricer (finds out at what level people wont fight us)

In [65]:
## from settings import settings

import numpy as np


class Logic():
    def __init__(self):
        self.config = settings()
        self.gamma = 0.01  # Risk aversion
        self.max_exposure = 0.8 * self.config.max_capital
    
    def generate_order_sizes(self, asset_price, asset_balance, current_exposure=0):
        """Generate optimal bid/ask sizes"""
        
        # Calculate prices
        bid_price = asset_price * (1 - self.config.bid_spread / 10000)
        ask_price = asset_price * (1 + self.config.ask_spread / 10000)
        
        # Base sizes from config
        base_size = self.config.min_order * self.config.max_capital / asset_price
        max_size = self.config.max_order * self.config.max_capital / asset_price
        
        # Risk adjustment - reduce sizes if high exposure
        exposure_factor = max(0.1, 1 - abs(current_exposure) / self.max_exposure)
        
        # Inventory skew - adjust based on current balance
        inventory_skew = np.tanh(asset_balance / (self.config.max_capital / asset_price))
        
        # Calculate optimal sizes
        bid_size = min(max_size, base_size * exposure_factor * (1 - inventory_skew * 0.5))
        ask_size = min(max_size, base_size * exposure_factor * (1 + inventory_skew * 0.5))
        
        return {
            'bid_size': max(0, round(bid_size, 6)),
            'ask_size': max(0, round(ask_size, 6)),
            'bid_price': round(bid_price, 6),
            'ask_price': round(ask_price, 6)
        }


Optimal Order Configuration:
Bid: 4.713160 @ 1.093905
Ask: 7.843917 @ 1.096095


In [66]:
Logic().generate_order_sizes(1.00,180)

{'bid_size': np.float64(6.418511),
 'ask_size': np.float64(13.581489),
 'bid_price': 0.999,
 'ask_price': 1.001}

In [ ]:
data = [Logic().generate_order_sizes(1.00,x) for x in range(1,201)]
from matplotlib import pyplot as plt

plt.plot(data['bid_size'],data['ask_size'])
plt.xlabel('Asset Value')
plt.ylabel('Currency Value')
plt.legend()

TypeError: list indices must be integers or slices, not str